# 05 Data structures and how the model is written
**Chapter 3: Methodology — C2 Data structures.** The Python twin of `05_data_structures.do`.

| Structure | Subscript | Example here |
|---|---|---|
| Cross-section | $y_i$ | 22 students, one semester |
| Time series | $y_t$ | one economy, 2000–2023 |
| Pooled cross-sections | $y_{it}$, new $i$ each $t$ | houses sold in 1978 and 1981 (simulated) |
| Panel | $y_{it}$, same $i$ every $t$ | 13 states, 2012–2022 (simulated) |

In [1]:
# Setup: install anything missing (needed on Google Colab), then load the libraries.
import importlib.util, subprocess, sys
for pkg in ["pandas", "numpy", "scipy", "statsmodels", "linearmodels", "matplotlib", "openpyxl"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

# Data live in ../data next to this notebook; online (e.g. Colab) they are read from GitHub.
DATA = Path("../data")
RAW_URL = "https://raw.githubusercontent.com/soonjanjan-lang/Chapter3-Methodology-lectnotes-dashboard/main/downloads/data/"

def load(name):
    p = DATA / name
    return pd.read_csv(p) if p.exists() else pd.read_csv(RAW_URL + name)

In [2]:
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS, RandomEffects
from scipy import stats

## 1. Cross-section: $Score_i = \beta_0 + \beta_1 StudyHours_i + \dots + u_i$

In [3]:
cs = load("ds_crosssection_students.csv")
cs["female"] = (cs["Gender"] == "Female").astype(int)
m1 = smf.ols("Final_Score ~ Study_Hours + Attendance + Sleep_Hours + PT_Job + HH_Income_k + female", cs).fit(cov_type="HC1")
print(m1.summary().tables[1])

                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      40.1358      8.514      4.714      0.000      23.449      56.823
Study_Hours     1.2128      0.177      6.867      0.000       0.867       1.559
Attendance      0.1225      0.136      0.899      0.369      -0.145       0.390
Sleep_Hours     1.3759      0.944      1.458      0.145      -0.474       3.226
PT_Job         -1.6464      2.784     -0.591      0.554      -7.103       3.810
HH_Income_k     0.1134      0.039      2.941      0.003       0.038       0.189
female          0.4407      2.892      0.152      0.879      -5.227       6.109


## 2. Time series: $Inflation_t = \beta_0 + \beta_1 Unemployment_t + \beta_2 Inflation_{t-1} + u_t$

In [4]:
ts = load("ds_timeseries_macro.csv").set_index("Year")
ts["Inflation_lag1"] = ts["Inflation"].shift(1)
m2 = smf.ols("Inflation ~ Unemployment + Inflation_lag1", ts.dropna()).fit()
m2_nw = smf.ols("Inflation ~ Unemployment + Inflation_lag1", ts.dropna()).fit(cov_type="HAC", cov_kwds={"maxlags": 2})
print(pd.DataFrame({"b": m2.params, "OLS se": m2.bse, "Newey-West se": m2_nw.bse}).round(4))
ts["gdp_growth"] = 100 * np.log(ts["GDP_USDb"]).diff()
print(ts["gdp_growth"].describe().round(2))

                     b  OLS se  Newey-West se
Intercept       6.3475  2.7782         2.1954
Unemployment   -1.1344  0.7276         0.6019
Inflation_lag1 -0.1257  0.2294         0.1322
count    23.00
mean      5.93
std       9.11
min     -13.29
25%       0.87
50%       8.33
75%      12.06
max      19.96
Name: gdp_growth, dtype: float64


## 3. Pooled cross-sections: difference-in-differences
$price_i = \beta_0 + \delta_0\, y81_i + \beta_1\, nearinc_i + \delta_1 (y81_i \times nearinc_i) + \text{controls} + u_i$ — the simulated true $\delta_1$ is −11,500.

In [5]:
pool = load("pooled_incinerator_sim.csv")
m3 = smf.ols("price ~ y81 * nearinc", pool).fit(cov_type="HC1")
m3c = smf.ols("price ~ y81 * nearinc + age + rooms + area", pool).fit(cov_type="HC1")
print(pd.DataFrame({"no controls": m3.params, "with controls": m3c.params}).round(1))
means = pool.pivot_table(index="nearinc", columns="y81", values="price", aggfunc="mean")
print(means.round(0))
print("DiD from the four means:", round((means.loc[1, 1] - means.loc[1, 0]) - (means.loc[0, 1] - means.loc[0, 0]), 2))

             no controls  with controls
Intercept        96377.4        22626.0
age                  NaN          -92.2
area                 NaN           12.0
nearinc         -17583.6       -17217.6
rooms                NaN         7731.6
y81              16736.9        15621.0
y81:nearinc     -11703.1       -11729.2
y81            0         1
nearinc                   
0        96377.0  113114.0
1        78794.0   83828.0
DiD from the four means: -11703.06


## 4. Panel: $saving_{it} = \beta_1 finlit_{it} + \beta_2 income_{it} + a_i + \lambda_t + u_{it}$ (true $\beta_1 = 0.15$)

In [6]:
pan = load("panel_states_sim.csv")
pan["covid"] = (pan["year"] == 2020).astype(int)
pan["trend"] = pan["year"]
panel = pan.set_index(["state", "year"])
pooled = smf.ols("saving_rate ~ finlit_index + income_k + C(year)", pan).fit(
    cov_type="cluster", cov_kwds={"groups": pd.factorize(pan["state"])[0]})
# the model you would report: state and year fixed effects, state-clustered standard errors
fe = PanelOLS.from_formula("saving_rate ~ finlit_index + income_k + EntityEffects + TimeEffects", panel).fit(
    cov_type="clustered", cluster_entity=True)
print(pd.DataFrame({"pooled OLS": pooled.params[["finlit_index", "income_k"]], "fixed effects": fe.params,
                    "FE clustered se": fe.std_errors}).round(4))

              pooled OLS  fixed effects  FE clustered se
finlit_index      0.3472         0.1494           0.0297
income_k          2.6489         1.1053           0.2619


### Fixed or random effects? The Hausman test
With only 13 states, the comparison uses a linear trend and a 2020 dummy instead of 10 year dummies so that
random effects is identified (the do-file does the same). H = (b_FE − b_RE)' [V_FE − V_RE]⁻¹ (b_FE − b_RE).

In [7]:
f = "saving_rate ~ finlit_index + income_k + trend + covid"
fe_h = PanelOLS.from_formula(f + " + EntityEffects", panel).fit()
re_h = RandomEffects.from_formula(f.replace("~", "~ 1 +"), panel).fit()
cols = ["finlit_index", "income_k", "trend", "covid"]
print(pd.DataFrame({"FE": fe_h.params[cols], "RE": re_h.params[cols]}).round(4))
b = (fe_h.params[cols] - re_h.params[cols]).values
V = (fe_h.cov.loc[cols, cols] - re_h.cov.loc[cols, cols]).values
H = float(b @ np.linalg.pinv(V) @ b)
print(f"Hausman chi2({len(cols)}) = {H:.2f}, p = {1 - stats.chi2.cdf(H, len(cols)):.4f}  → reject RE: use fixed effects")
print("(Stata's 'sigmamore' option scales the variances slightly differently; the conclusion is the same.)")

                  FE      RE
finlit_index  0.1360  0.1684
income_k      1.0426  1.4535
trend         0.0420 -0.0429
covid        -2.6896 -2.6295


Hausman chi2(4) = -58.84, p = 1.0000  → reject RE: use fixed effects
(Stata's 'sigmamore' option scales the variances slightly differently; the conclusion is the same.)


In [8]:
assert abs(m3c.params["y81:nearinc"] - (-11729.18)) < 1
assert abs(fe.params["finlit_index"] - 0.1494) < 0.0005
print("DiD and fixed-effects estimates match the Stata do-file.")

DiD and fixed-effects estimates match the Stata do-file.
